In [ ]:
import os
import pandas as pd
import glob
from tqdm import tqdm

# 获取当前文件夹路径
folder_path = os.getcwd()

# 获取所有以 "data" 开头且后接数字的 Excel 文件
excel_files = glob.glob(os.path.join(folder_path, "data*.xlsx"))

# 初始化用于存储结果的列表
results = []

# 使用 tqdm 为遍历文件添加进度条
for file in tqdm(excel_files, desc="Processing files"):
    try:
        # 读取 Excel 文件
        df = pd.read_excel(file)

        # 检查是否存在第三列
        if df.shape[1] >= 3:
            # 获取第三列的数据
            col_3 = df.iloc[:, 2]

            # 查找第三列中不等于 1 和 3 的值，并替换为 3
            invalid_rows = col_3[~col_3.isin([1, 3])]

            # 如果找到异常值，进行替换操作
            df.iloc[:, 2] = df.iloc[:, 2].apply(lambda x: 3 if x not in [1, 3] else x)

            # 如果找到异常值，记录文件名和行号
            if not invalid_rows.empty:
                for index, value in invalid_rows.items():
                    results.append((file, index + 1, value))  # 行号加1以匹配Excel中的行号

            # 保存修改后的文件
            df.to_excel(file, index=False)
            print(f"文件 {file} 已更新")

    except Exception as e:
        tqdm.write(f"处理文件 {file} 时出错: {e}")

# 输出统计结果
if results:
    print("以下是包含异常值的文件及行号：")
    for result in results:
        print(f"文件: {result[0]}，行号: {result[1]}，值: {result[2]}")
else:
    print("所有文件的第三列数据均为 1 或 3。")
